# Winter wheat phenology — TorchCrop against the SAGE crop calendar (gridded)

**What this notebook does.** It bins the simulated sowing and maturity dates
onto the 0.5° grid of the Sacks et al. (2010) SAGE crop calendar and compares
them per cell — the gridded counterpart of
[`phenology_evaluation.ipynb`](phenology_evaluation.ipynb), with one important
difference: **SAGE reports a planting date**, not a green-up proxy that
changes meaning with climate the way CyBench's `sos` does. This is therefore
the more direct test of `config.SIM_SOWING_DOY`, the constant every cell in
the run is sown on.

## What each side contains

| Stage | TorchCrop | SAGE | Comparable? |
|---|---|---|---|
| Sowing | **constant input**, DOY 270 for every cell | `plant`, with a `plant.start`-`plant.end` window | yes — a genuine test of the constant |
| Flowering | **not in the run** | not in the calendar | no |
| Maturity | first day at `DVS ≥ 2` | `harvest`, with a `harvest.start`-`harvest.end` window | yes, with a harvest-after-maturity offset expected |

* **SAGE is a climatology.** One planting and harvest date per cell, assembled
  around 1990-2000 from agricultural-census and extension-service reports —
  no year dimension, so nothing here tests interannual skill, only whether the
  model places the *average* season in the right week.
* **A third of the domain is filled, not reported.** Where no reporting unit
  covers a cell, the `.fill` product extrapolates from the nearest one; those
  cells carry their donor's date exactly and are flagged `filled = True`
  throughout this notebook so they can be read separately.
* **A date comes with a window**, 10 to 100 days wide in Europe. Whether the
  simulated date falls **inside** that window is a more forgiving and more
  informative test than a bias in days, computed with `doy.doy_in_window`.

All reusable code is in [`utils/`](utils/); this notebook is the workflow only.

## 0. Setup

In [ ]:
import logging
import sys
from pathlib import Path

# cropmodelling4eu is installed (pip install -e .), so the evaluation
# library is imported like any other package rather than off sys.path.

import numpy as np
import pandas as pd

from cropmodelling4eu.evaluation import config, doy, grid, metrics, plots, torchcrop
from cropmodelling4eu.evaluation.style import use_style

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s",
                    force=True)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

PALETTE = use_style("light")
config.ensure_output_dirs()

pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 140)

print(f"TorchCrop run : {config.TORCHCROP_RUN_DIR}")
print(f"Outputs       : {config.OUTPUT_DIR}")
print(f"Grid          : {config.GRID_RES_DEG}° over {config.EUROPE_BBOX}")

## 1. Load the simulation and derive the stage dates, binned onto the SAGE grid

In [ ]:
sim = torchcrop.load_simulation(columns=[
    "SimplaceID", "year", "lon", "lat", "days_to_maturity", "yield_t_ha",
    "biomass_g_m2", "max_lai", "tranrf_mean", "nni_mean", "heat_stress_factor",
])
sim = grid.crop_to_bbox(sim, config.EUROPE_BBOX)
# Tagged onto every 10 km row (not just the binned means) so the diagnostics
# in §7 can group the run's own state variables by 0.5° cell too.
sim["grid_id"] = grid.grid_cell_id(*grid.snap_to_grid(sim["lon"], sim["lat"]))

sim_grid = grid.bin_cells(
    sim, {"sowing_doy": True, "maturity_doy": True, "season_length_days": False},
)
# `season_length_days` exists on both sides once paired with the SAGE calendar
# (its own `tot.days`); rename the simulated one now rather than disambiguate
# a pandas merge suffix later.
sim_grid = sim_grid.rename(columns={"season_length_days": "sim_season_length_days"})

print(f"{len(sim):,} cell-seasons on the 10 km grid -> "
      f"{len(sim_grid):,} 0.5° cells, "
      f"{sim_grid['n_cells'].median():.0f} simulated cells per cell (median)")
sim_grid.describe().T

## 2. Load the SAGE calendar

In [ ]:
from cropmodelling4eu.evaluation import sage

calendar = sage.load_sage_calendar(crop=config.SAGE_CROP)
print(f"SAGE crop: {config.SAGE_CROP} ({config.SAGE_ROOT})")
print(f"{len(calendar):,} cells in the domain, "
      f"{100 * calendar['filled'].mean():.0f}% filled from a neighbouring unit")
calendar.describe().T

## 3. Pair the two sides

A plain join on `grid_id` — SAGE carries no year, so every simulated season in
a cell is compared against the same climatological date, exactly as CyBench's
`sos`/`eos` are in the country-level notebook.

In [ ]:
paired = grid.pair_gridded(sim_grid, calendar, ["grid_id"])

for stage in config.CALENDAR_STAGES:
    paired[f"{stage.key}_error"] = doy.doy_difference(
        paired[stage.sim_col], paired[stage.obs_col]
    )
    paired[f"{stage.key}_in_window"] = doy.doy_in_window(
        paired[stage.sim_col], paired[stage.start_col], paired[stage.end_col]
    )

print(f"{len(paired):,} paired 0.5° cells, "
      f"{100 * paired['filled'].mean():.0f}% filled")
paired.head()

## 4. Metrics by stage

Reported once pooled and once restricted to the reported (non-filled) cells —
the filled third of the domain repeats a neighbour's exact date, which can
only make a local bias look smoother than it is.

In [ ]:
pooled = {}
reported = paired[~paired["filled"]]
for stage in config.CALENDAR_STAGES:
    pooled[stage.label] = metrics.phenology_metrics(paired[stage.obs_col], paired[stage.sim_col])
    pooled[f"{stage.label} (reported only)"] = metrics.phenology_metrics(
        reported[stage.obs_col], reported[stage.sim_col]
    )

pooled_table = pd.DataFrame(pooled).T[list(metrics.PHENOLOGY_METRIC_ORDER)]
pooled_table["observed date"] = pooled_table["obs_mean"].map(doy.doy_to_month_day)
pooled_table["simulated date"] = pooled_table["sim_mean"].map(doy.doy_to_month_day)
pooled_table.round(2)

In [ ]:
for stage in config.CALENDAR_STAGES:
    print(f"--- {stage.label} ---")
    print(stage.caveat)
    print()

### Window containment

The share of cells whose simulated date falls inside the SAGE window, split
into "too early", "inside" and "too late" — the categorical answer the bias
figures below cannot give on their own.

In [ ]:
window_summary = {}
for stage in config.CALENDAR_STAGES:
    position = doy.window_position(paired[stage.sim_col], paired[stage.start_col],
                                   paired[stage.end_col])
    category = np.select(
        [position < 0, position == 0, position > 0],
        ["before window", "in window", "after window"],
        default="not paired",
    )
    paired[f"{stage.key}_position"] = category
    counts = pd.Series(category).value_counts(normalize=True) * 100
    window_summary[stage.label] = counts

pd.DataFrame(window_summary).round(1).fillna(0.0)

## 5. Figures

### 5.1 Observed against simulated, one panel per stage

In [ ]:
for stage in config.CALENDAR_STAGES:
    fig = plots.scatter_density(
        paired, stage.obs_col, stage.sim_col, circular=True,
        stats=pooled[stage.label], metric_keys=("n", "bias", "rmse", "pearson_r"),
        title=f"{stage.label}: simulated against SAGE '{stage.obs_col}'",
        xlabel=f"SAGE {stage.obs_col}", ylabel=f"TorchCrop {stage.label.lower()}",
    )
    plots.save(fig, f"sage_01_scatter_{stage.key}")
    display(fig)

### 5.2 Bias maps

In [ ]:
for stage in config.CALENDAR_STAGES:
    stage_paired = paired.dropna(subset=[f"{stage.key}_error"])
    fig = plots.cell_map(
        stage_paired, f"{stage.key}_error",
        title=f"{stage.label} bias (simulated - observed)",
        cbar_label="Bias (days; positive = late)", diverging=True,
    )
    plots.save(fig, f"sage_02_map_bias_{stage.key}")
    display(fig)

### 5.3 Window containment maps

Discrete fills rather than a colour ramp: "inside the window" has no useful
distance to "before" or "after" it.

In [ ]:
for stage in config.CALENDAR_STAGES:
    fig = plots.category_map(
        paired, f"{stage.key}_position",
        categories=["before window", "in window", "after window"],
        title=f"{stage.label}: simulated date against the SAGE window",
        legend_title=f"vs. {stage.start_col}-{stage.end_col}",
    )
    plots.save(fig, f"sage_03_map_window_{stage.key}")
    display(fig)

### 5.4 Season length

`sim_season_length_days` is sowing-to-maturity; SAGE's `season_length_days` is
planting-to-harvest. The two are not the same interval — this compares their
*durations*, not the dates that bound them, so it survives the DOY 270 sowing
offset entirely.

In [ ]:
season_pooled = metrics.yield_metrics(
    paired["season_length_days"], paired["sim_season_length_days"]
)
fig = plots.scatter_density(
    paired, "season_length_days", "sim_season_length_days", stats=season_pooled,
    metric_keys=("n", "bias", "rmse", "pearson_r"),
    title="Season length: simulated (sowing-to-maturity) against SAGE (planting-to-harvest)",
    xlabel="SAGE season length (days)", ylabel="TorchCrop season length (days)",
)
plots.save(fig, "sage_04_scatter_season_length")
fig

## 6. Error decomposition — how much of the maturity error is the sowing offset

The run's maturity date is `sowing_doy + days_to_maturity`. If sowing is early
by `X` days and the simulated season length is otherwise right, maturity is
early by about `X` days too — a sowing artefact, not a growth-rate error. This
splits the maturity bias into the two additive pieces so the difference is
visible rather than absorbed into one number.

In [ ]:
# `paired` is already one row per grid_id (SAGE carries no year), so this is a
# rename, not an aggregation.
decomposition = pd.DataFrame({
    "Sowing offset": paired["sowing_error"].to_numpy(),
    "Season-length residual": (paired["maturity_error"] - paired["sowing_error"]).to_numpy(),
}, index=paired["grid_id"].to_numpy())
decomposition.index.name = "grid_id"
decomposition["Maturity error (total)"] = (
    decomposition["Sowing offset"] + decomposition["Season-length residual"]
)

print("Mean absolute contribution to the maturity error:")
print(decomposition[["Sowing offset", "Season-length residual"]].abs().mean().round(2).to_string())

fig = plots.component_bars(
    decomposition.sample(min(40, len(decomposition)), random_state=0)
                .sort_values("Maturity error (total)"),
    ["Sowing offset", "Season-length residual"],
    title="Maturity-error decomposition, 40 sampled cells (sorted by total error)",
    xlabel="Days (positive = late)",
)
plots.save(fig, "sage_05_bars_decomposition")
fig

## 7. Diagnostics — the run's own state next to the bias

In [ ]:
state = (
    sim.groupby("grid_id", as_index=False)
    .agg(lon=("lon", "first"), lat=("lat", "first"),
         max_lai=("max_lai", "mean"), biomass=("biomass_g_m2", "mean"),
         days_to_maturity=("days_to_maturity", "mean"),
         water_stress=("tranrf_mean", "mean"), n_index=("nni_mean", "mean"),
         heat=("heat_stress_factor", "mean"),
         failed_share=("yield_t_ha", lambda s: float((s < 0.5).mean())))
)

state["maturity_bias"] = state["grid_id"].map(paired.set_index("grid_id")["maturity_error"])
state["sowing_bias"] = state["grid_id"].map(paired.set_index("grid_id")["sowing_error"])
state = state.dropna(subset=["maturity_bias"])

print("Correlation of cell maturity bias with the run's own state variables:")
print(state.corr(numeric_only=True)["maturity_bias"]
      .drop(["maturity_bias", "sowing_bias"]).round(2).sort_values().to_string())
state.round(2).head()

## 8. Summary

In [ ]:
summary = pd.Series({
    "0.5-degree cells": paired["grid_id"].nunique(),
    "filled share (%)": round(100 * paired["filled"].mean(), 1),
    "sowing bias (days)": round(pooled["Sowing"]["bias"], 1),
    "sowing RMSE (days)": round(pooled["Sowing"]["rmse"], 1),
    "sowing in-window (%)": round(
        100 * paired["sowing_in_window"].mean(), 1),
    "maturity bias (days)": round(pooled["Maturity / harvest"]["bias"], 1),
    "maturity RMSE (days)": round(pooled["Maturity / harvest"]["rmse"], 1),
    "maturity Pearson r": round(pooled["Maturity / harvest"]["pearson_r"], 2),
    "season-length bias (days)": round(season_pooled["bias"], 1),
    "season-length RMSE (days)": round(season_pooled["rmse"], 1),
}, name="value")

paired.to_csv(config.TABLE_DIR / "sage_paired_cell.csv", index=False, float_format="%.4f")
pooled_table.to_csv(config.TABLE_DIR / "sage_metrics_pooled.csv", float_format="%.3f")
state.to_csv(config.TABLE_DIR / "sage_diagnostics_by_cell.csv", float_format="%.4f")
print(f"tables written to {config.TABLE_DIR}")
print(f"figures written to {config.FIGURE_DIR}")
summary.to_frame()

### What the numbers mean, and what they do not

* **Sowing is the genuine result here, unlike in the CyBench notebook.** SAGE
  `plant` does not switch convention with climate the way CyBench `sos` does,
  so this bias is a direct, continent-wide statement about the DOY 270 latch —
  and §5.3/§6 say where and by how much a real sowing calendar would have to
  differ from that constant.
* **The window test and the bias are complementary, not redundant.** A cell can
  be "in window" with a large bias if the window is wide, and "out of window"
  with a small one if the window is narrow — read both.
* **Maturity still carries the CyBench notebook's caveat**: SAGE `harvest` is
  at or after `DVS = 2`, so part of the bias is that interval, not model error.
  §6's decomposition only separates the sowing contribution from the rest; it
  does not remove the harvest-after-maturity gap.
* **A third of the domain is a repeated neighbour's date, not an independent
  observation.** The `filled` split in §4 is there so a result is not quoted
  from a part of the map that is smoother than the underlying data.
* **The reference is climatological**, exactly as CyBench's calendar is:
  nothing here tests interannual timing, only the average placement of the
  season.